# ClarIA — Pipeline: N estados de cuenta → Tablas DB

```
PDFs → pdfplumber (texto) → Claude Haiku (JSON) → tarjetas / transacciones / compras_msi
```

**Lo que hace este pipeline:**
1. Procesa N PDFs de BBVA (misma tarjeta o distintas)
2. Identifica el último estado por tarjeta → saldo actual correcto
3. Deduplica transacciones que aparecen en varios periodos
4. Genera SQL listo para correr en la DB de producción

**No hace (viene después):** categorización — se hará con embeddings.

## 0. Setup

In [ ]:
%pip install anthropic pdfplumber python-dotenv pandas --quiet

In [ ]:
import pdfplumber
import anthropic
import json
import os
import re
import glob
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from dotenv import load_dotenv
import pandas as pd

load_dotenv(Path('..') / '.env')
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
print(f'✅ Listo  |  key: {os.getenv("ANTHROPIC_API_KEY", "NO ENCONTRADA")[:16]}...')

## 1. Config

In [ ]:
PDF_DIR         = 'bbva'                    # directorio con los PDFs
MODEL           = 'claude-haiku-4-5-20251001'
MAX_CHARS       = 12000
MAX_TOKENS_RESP = 4096
USUARIO_ID      = 1                         # id del usuario en la DB

CARD_COLORS = [
    ('#002C7A', '#0058C8'),
    ('#880000', '#CC1E00'),
    ('#003F6B', '#006EA8'),
    ('#1A4731', '#276749'),
    ('#2D1B69', '#5B2D8E'),
]

# Meses en español para convertir fechas BBVA
MESES_ES   = {'ene':1,'feb':2,'mar':3,'abr':4,'may':5,'jun':6,
              'jul':7,'ago':8,'sep':9,'oct':10,'nov':11,'dic':12}
MESES_ABBR = {v: k for k, v in MESES_ES.items()}

## 2. Prompt y funciones

In [ ]:
PROMPT_TEMPLATE = """Eres un extractor de datos de estados de cuenta de tarjeta de crédito BBVA México.
Del siguiente texto extrae TODOS los campos y responde SOLO con JSON válido, sin markdown, sin texto extra.
Estructura exacta a devolver:
{{
  "nombre_tarjeta": "nombre del producto (ej: BBVA Dorada, Azul, Platino, Clásica)",
  "banco": "BBVA",
  "last4": "últimos 4 dígitos del número de tarjeta (string)",
  "limite": 0,
  "saldo_usado": 0,
  "saldo_pagar": 0,
  "dia_corte": 15,
  "dia_pago": 10,
  "periodo": "DD-MMM-YYYY al DD-MMM-YYYY",
  "transacciones_msi": [
    {{
      "fecha_operacion": "DD-MMM-YYYY",
      "descripcion": "descripción tal como aparece",
      "monto_original": 0,
      "saldo_pendiente": 0,
      "pago_requerido": 0,
      "num_pago_actual": 4,
      "num_pagos_total": 12,
      "tasa_interes": 0.00
    }}
  ],
  "transacciones_regulares": [
    {{
      "fecha_operacion": "DD-MMM-YYYY",
      "fecha_cargo": "DD-MMM-YYYY",
      "descripcion": "descripción tal como aparece",
      "monto": 0,
      "tipo": "abono"
    }}
  ]
}}

─── REGLAS GENERALES ───────────────────────────────────────────────────────────
- limite: línea de crédito total en MXN (número, sin comas ni $)
- saldo_usado: saldo al corte o total a pagar en MXN (número, sin comas)
- dia_corte / dia_pago: día del mes como entero (1-31)
- Si un campo no aparece, usa null
- Todas las fechas en formato DD-MMM-YYYY (ej: 10-feb-2026)

─── REGLAS: transacciones_msi ──────────────────────────────────────────────────
Sección: "COMPRAS Y CARGOS DIFERIDOS A MESES SIN INTERESES"
- monto_original: columna "Monto original" (número, sin comas ni $)
- saldo_pendiente: columna "Saldo pendiente" (número, sin comas ni $)
- pago_requerido: columna "Pago requerido" (número, sin comas ni $)
- num_pago_actual / num_pagos_total: de "Núm. de pago" (ej: "4 de 12" → 4 y 12)
- tasa_interes: columna "Tasa de interés aplicable" como decimal (ej: "0.00%" → 0.00)

─── REGLAS: transacciones_regulares ────────────────────────────────────────────
Sección: "CARGOS, COMPRAS Y ABONOS REGULARES (NO A MESES)"
- monto: positivo si cargo (+), negativo si abono (-)
- tipo: "cargo" si monto > 0, "abono" si monto < 0
- descripcion: solo la primera línea del movimiento

Texto del estado de cuenta:
{texto}"""

In [ ]:
# ── Helpers de fecha ─────────────────────────────────────────────────────────

def parse_fecha_bbva(s: str) -> datetime:
    """'09-mar-2026' → datetime(2026, 3, 9)"""
    d, m, y = s.strip().lower().split('-')
    return datetime(int(y), MESES_ES[m], int(d))

def fecha_corte_del_periodo(periodo: str) -> datetime:
    """'10-feb-2026 al 09-mar-2026' → datetime del día de corte"""
    return parse_fecha_bbva(periodo.split(' al ')[-1].strip())

def bbva_a_iso(s) -> str | None:
    """'09-mar-2026' → '2026-03-09' (formato DATE de PostgreSQL)"""
    if not s:
        return None
    try:
        return parse_fecha_bbva(s).strftime('%Y-%m-%d')
    except Exception:
        return None

def periodo_db(fecha_corte: datetime) -> str:
    """datetime → 'mar-2026'  (formato del dashboard)"""
    return f"{MESES_ABBR[fecha_corte.month]}-{fecha_corte.year}"

# ── Tipo de transacción para la DB ───────────────────────────────────────────
# La DB usa:  'regular'   → cargos normales (mostrados en el dashboard)
#             'msi_cuota' → cuotas de planes MSI (filtradas del dashboard)
#             'abono'     → pagos a la tarjeta (filtrados del dashboard)

def tipo_db(tx: dict) -> str:
    desc = (tx.get('descripcion') or '').upper()
    if re.match(r'^\d+\s+DE\s+\d+', desc):   # patrón '04 DE 12 PLAN X'
        return 'msi_cuota'
    if tx.get('tipo') == 'abono':
        return 'abono'
    return 'regular'

# ── Normalización para deduplicación ─────────────────────────────────────────

def normalizar(s: str) -> str:
    s = re.sub(r'\s*;\s*Tarjeta Digital \*+\d+', '', s, flags=re.IGNORECASE)
    return s.strip().upper()

# ── Extracción de texto y llamada a Claude ───────────────────────────────────

def extraer_texto_pdf(path: str) -> str:
    with pdfplumber.open(path) as pdf:
        return '\n\n'.join(
            f'--- Página {i+1} ---\n{p.extract_text() or ""}'
            for i, p in enumerate(pdf.pages)
        )

def extraer_info_tarjeta(texto: str) -> tuple[dict, object]:
    response = client.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS_RESP,
        messages=[{'role': 'user', 'content': PROMPT_TEMPLATE.format(texto=texto[:MAX_CHARS])}]
    )
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', response.content[0].text.strip())
    return json.loads(raw), response.usage

print('✅ Funciones listas')

## 3. Procesar todos los PDFs

Corre una sola vez. El resultado queda en `raw_estados`.

In [ ]:
pdfs = sorted(glob.glob(f'{PDF_DIR}/*.pdf'))
print(f'📂 {len(pdfs)} PDFs en ./{PDF_DIR}/\n')

raw_estados, errores = [], []

for path in pdfs:
    nombre = Path(path).name
    print(f'  ⏳ {nombre}', end='  ')
    try:
        texto = extraer_texto_pdf(path)
        info, usage = extraer_info_tarjeta(texto)
        info['_archivo'] = nombre
        info['_tokens']  = usage.input_tokens + usage.output_tokens
        try:
            info['_fecha_corte'] = fecha_corte_del_periodo(info['periodo'])
        except Exception:
            info['_fecha_corte'] = datetime(2000, 1, 1)
        raw_estados.append(info)
        print(f'✅  {info.get("nombre_tarjeta")} ****{info.get("last4")}  '
              f'{info.get("periodo")}  {usage.input_tokens+usage.output_tokens} tok')
    except Exception as e:
        errores.append({'archivo': nombre, 'error': str(e)})
        print(f'❌  {e}')

print(f'\n✅ {len(raw_estados)} ok   ❌ {len(errores)} errores'
      f'   💰 {sum(e["_tokens"] for e in raw_estados):,} tokens totales')

## 4. Identificar tarjetas únicas y último estado

In [ ]:
por_tarjeta = defaultdict(list)
for e in raw_estados:
    por_tarjeta[e['last4']].append(e)

for last4 in por_tarjeta:
    por_tarjeta[last4].sort(key=lambda x: x['_fecha_corte'])

# Iterar estados en orden cronológico global (para dedup)
estados_cronologicos = sorted(raw_estados, key=lambda x: x['_fecha_corte'])

print(f'Tarjetas únicas: {len(por_tarjeta)}\n')
for last4, estados in por_tarjeta.items():
    u = estados[-1]
    print(f'  ****{last4}  {u["nombre_tarjeta"]:25s}  '
          f'{len(estados)} estado(s)  '
          f'último: {u["periodo"]}  '
          f'saldo=${u.get("saldo_usado") or 0:,.2f}')

## 5. Tabla `tarjetas`

In [ ]:
tarjeta_id_map = {}
filas_tarjetas = []

for idx, (last4, estados) in enumerate(sorted(por_tarjeta.items()), start=1):
    tarjeta_id_map[last4] = idx
    u     = estados[-1]   # último estado = datos más recientes
    c1,c2 = CARD_COLORS[(idx-1) % len(CARD_COLORS)]
    filas_tarjetas.append({
        'tarjeta_id'  : idx,
        'usuario_id'  : USUARIO_ID,
        'nombre'      : u['nombre_tarjeta'],
        'banco'       : u['banco'],
        'last4'       : last4,
        'limite'      : u.get('limite')      or 0,
        'saldo_usado' : u.get('saldo_usado') or 0,
        'dia_corte'   : u.get('dia_corte')   or 15,
        'dia_pago'    : u.get('dia_pago')    or 10,
        'color_inicio': c1,
        'color_fin'   : c2,
        # metadata (no van a la DB)
        '_num_estados'   : len(estados),
        '_periodo_inicio': estados[0]['periodo'],
        '_periodo_fin'   : estados[-1]['periodo'],
    })

df_tarjetas = pd.DataFrame(filas_tarjetas)
cols_db = ['tarjeta_id','usuario_id','nombre','banco','last4','limite','saldo_usado','dia_corte','dia_pago','color_inicio','color_fin']
print(f'🏦 tarjetas — {len(df_tarjetas)} fila(s)\n')
df_tarjetas[cols_db + ['_num_estados','_periodo_inicio','_periodo_fin']]

## 6. Tabla `transacciones`

Columnas: `usuario_id, tarjeta_id, fecha_operacion, fecha_cargo, descripcion, monto_mxn, tipo, periodo, via`

- `monto_mxn` → **siempre positivo** (el dashboard lo invierte en UI)
- `tipo` → `'regular'` (cargo normal) · `'msi_cuota'` (cuota MSI) · `'abono'` (pago a tarjeta)
- `periodo` → `'mar-2026'` (mes del corte, formato del dashboard)
- `categoria` → pendiente, se asignará con embeddings

In [ ]:
vistos   = set()
filas_tx = []

for estado in estados_cronologicos:
    last4      = estado['last4']
    tarjeta_id = tarjeta_id_map[last4]
    per_db     = periodo_db(estado['_fecha_corte'])

    for tx in (estado.get('transacciones_regulares') or []):
        clave = (
            tarjeta_id,
            tx.get('fecha_operacion', ''),
            normalizar(tx.get('descripcion', '')),
            abs(tx.get('monto', 0))
        )
        if clave in vistos:
            continue
        vistos.add(clave)

        filas_tx.append({
            'usuario_id'     : USUARIO_ID,
            'tarjeta_id'     : tarjeta_id,
            'fecha_operacion': bbva_a_iso(tx.get('fecha_operacion')),
            'fecha_cargo'    : bbva_a_iso(tx.get('fecha_cargo')),
            'descripcion'    : tx.get('descripcion'),
            'monto_mxn'      : abs(tx.get('monto', 0)),   # ← siempre positivo
            'tipo'           : tipo_db(tx),                # 'regular'/'abono'/'msi_cuota'
            'periodo'        : per_db,                     # 'mar-2026'
            'via'            : 'pdf',
        })

df_tx = pd.DataFrame(filas_tx)
print(f'💳 transacciones — {len(df_tx)} registros únicos\n')
tipos = df_tx['tipo'].value_counts().to_dict() if len(df_tx) else {}
for t, n in tipos.items():
    print(f'   {t:<12} {n}')
df_tx.head(10)

## 7. Tabla `compras_msi`

Columnas: `usuario_id, tarjeta_id, descripcion, comercio, monto_total, total_pagos, pagos_hechos, cuota_mensual, fecha_inicio, activo`

Deduplicación: mismo plan aparece en cada estado con `num_pago_actual` distinto → tomar el más reciente.

In [ ]:
msi_map = {}

for estado in estados_cronologicos:
    last4      = estado['last4']
    tarjeta_id = tarjeta_id_map[last4]

    for msi in (estado.get('transacciones_msi') or []):
        clave = (tarjeta_id, normalizar(msi.get('descripcion','')), msi.get('monto_original', 0))
        prev  = msi_map.get(clave)
        if prev is None or (msi.get('num_pago_actual') or 0) > (prev['pagos_hechos'] or 0):
            msi_map[clave] = {
                'usuario_id'    : USUARIO_ID,
                'tarjeta_id'    : tarjeta_id,
                'descripcion'   : msi.get('descripcion'),
                'comercio'      : None,   # no disponible en PDF
                'monto_total'   : msi.get('monto_original', 0),
                'total_pagos'   : msi.get('num_pagos_total', 1),
                'pagos_hechos'  : msi.get('num_pago_actual', 0),
                'cuota_mensual' : round(
                    msi.get('monto_original', 0) / max(msi.get('num_pagos_total', 1), 1), 2
                ),
                'fecha_inicio'  : bbva_a_iso(msi.get('fecha_operacion')),
                'activo'        : True,
            }

df_msi = pd.DataFrame(list(msi_map.values())) if msi_map else pd.DataFrame()
print(f'🔒 compras_msi — {len(df_msi)} planes únicos\n')
if len(df_msi):
    cols = ['tarjeta_id','descripcion','monto_total','total_pagos','pagos_hechos','cuota_mensual','fecha_inicio']
    print(df_msi[cols].to_string(index=False))

## 8. Verificación

In [ ]:
print('═' * 60)
print('RESUMEN')
print('═' * 60)
print(f'  tarjetas       : {len(df_tarjetas)}')
print(f'  transacciones  : {len(df_tx)}')
if len(df_tx):
    r = df_tx[df_tx['tipo']=='regular']
    print(f'    regular      : {len(r)}  |  total: ${r["monto_mxn"].sum():,.2f}')
    print(f'    msi_cuota    : {(df_tx["tipo"]=="msi_cuota").sum()}')
    print(f'    abono        : {(df_tx["tipo"]=="abono").sum()}')
    print(f'    rango fechas : {df_tx["fecha_operacion"].min()} → {df_tx["fecha_operacion"].max()}')
print(f'  compras_msi    : {len(df_msi)}')

print('\n── Por tarjeta ──────────────────────────────────────────')
for _, row in df_tarjetas.iterrows():
    tid = row['tarjeta_id']
    n_r   = len(df_tx[(df_tx['tarjeta_id']==tid)&(df_tx['tipo']=='regular')]) if len(df_tx) else 0
    n_msi = len(df_msi[df_msi['tarjeta_id']==tid]) if len(df_msi) else 0
    print(f'  [{tid}] {row["nombre"]:25s} ****{row["last4"]}  '
          f'{n_r} cargos  {n_msi} MSI  '
          f'saldo=${row["saldo_usado"]:,.2f}  límite=${row["limite"]:,.2f}')

## 9. Export CSV

In [ ]:
OUT = Path('output')
OUT.mkdir(exist_ok=True)

cols_tarjetas = ['usuario_id','nombre','banco','last4','limite','saldo_usado','dia_corte','dia_pago','color_inicio','color_fin']
cols_tx       = ['usuario_id','tarjeta_id','fecha_operacion','fecha_cargo','descripcion','monto_mxn','tipo','periodo','via']
cols_msi      = ['usuario_id','tarjeta_id','descripcion','comercio','monto_total','total_pagos','pagos_hechos','cuota_mensual','fecha_inicio','activo']

df_tarjetas[cols_tarjetas].to_csv(OUT / 'tarjetas.csv', index=False)
df_tx[cols_tx].to_csv(OUT / 'transacciones.csv', index=False)
if len(df_msi):
    df_msi[cols_msi].to_csv(OUT / 'compras_msi.csv', index=False)

print('✅ CSVs en ./output/')
print(f'   tarjetas.csv       {len(df_tarjetas)} filas')
print(f'   transacciones.csv  {len(df_tx)} filas')
print(f'   compras_msi.csv    {len(df_msi)} filas')

## 10. SQL para producción

El SQL usa **subqueries por `last4`** en lugar de IDs hardcodeados, así funciona
sin importar qué IDs asigne el SERIAL de la DB.

**Orden de ejecución:** tarjetas → transacciones → compras_msi

In [ ]:
def v(val):
    """Escapa un valor para SQL."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return 'NULL'
    if isinstance(val, bool):
        return 'TRUE' if val else 'FALSE'
    if isinstance(val, str):
        return "'" + val.replace("'", "''") + "'"
    return str(val)

def tarjeta_subquery(last4: str) -> str:
    """Subquery que resuelve tarjeta_id en runtime."""
    return f"(SELECT id FROM tarjetas WHERE last4={v(str(last4))} AND usuario_id={USUARIO_ID} LIMIT 1)"

print('✅ Helpers SQL listos')

In [ ]:
lineas = [
    '-- ══════════════════════════════════════════════════════════',
    '-- 1. TARJETAS',
    '-- ══════════════════════════════════════════════════════════',
]
for _, row in df_tarjetas.iterrows():
    lineas.append(
        f"INSERT INTO tarjetas (usuario_id,nombre,banco,last4,limite,saldo_usado,dia_corte,dia_pago,color_inicio,color_fin,activo) "
        f"VALUES ({USUARIO_ID},{v(row['nombre'])},{v(row['banco'])},{v(str(row['last4']))},"
        f"{row['limite']},{row['saldo_usado']},{row['dia_corte']},{row['dia_pago']},"
        f"{v(row['color_inicio'])},{v(row['color_fin'])},TRUE);"
    )

sql_tarjetas = '\n'.join(lineas)
(OUT / 'insert_tarjetas.sql').write_text(sql_tarjetas)
print(sql_tarjetas)
print(f'\n✅ output/insert_tarjetas.sql')

In [ ]:
lineas = [
    '-- ══════════════════════════════════════════════════════════',
    '-- 2. TRANSACCIONES',
    f'-- {len(df_tx)} registros  (regular / msi_cuota / abono)',
    '-- ══════════════════════════════════════════════════════════',
]
for _, row in df_tx.iterrows():
    tid_sql = tarjeta_subquery(str(int(row['tarjeta_id'])).zfill(4))  # resolverá por last4
    # Necesitamos el last4 de este tarjeta_id
    last4_row = df_tarjetas[df_tarjetas['tarjeta_id']==row['tarjeta_id']]['last4'].values[0]
    tid_sql   = tarjeta_subquery(last4_row)

    lineas.append(
        f"INSERT INTO transacciones "
        f"(usuario_id,tarjeta_id,fecha_operacion,fecha_cargo,descripcion,monto_mxn,tipo,periodo,via) "
        f"SELECT {USUARIO_ID},{tid_sql},{v(row['fecha_operacion'])},{v(row['fecha_cargo'])},"
        f"{v(row['descripcion'])},{row['monto_mxn']},{v(row['tipo'])},{v(row['periodo'])},{v(row['via'])} "
        f"WHERE NOT EXISTS (SELECT 1 FROM transacciones WHERE usuario_id={USUARIO_ID} "
        f"AND fecha_operacion={v(row['fecha_operacion'])} "
        f"AND descripcion={v(row['descripcion'])} "
        f"AND monto_mxn={row['monto_mxn']});"
    )

sql_tx = '\n'.join(lineas)
(OUT / 'insert_transacciones.sql').write_text(sql_tx)
print(f'✅ {len(df_tx)} transacciones → output/insert_transacciones.sql')
print('\nPrimeras 3:')
print('\n'.join(lineas[4:7]))

In [ ]:
if not len(df_msi):
    print('Sin planes MSI.')
else:
    lineas = [
        '-- ══════════════════════════════════════════════════════════',
        '-- 3. COMPRAS MSI',
        '-- ══════════════════════════════════════════════════════════',
    ]
    for _, row in df_msi.iterrows():
        last4_row = df_tarjetas[df_tarjetas['tarjeta_id']==row['tarjeta_id']]['last4'].values[0]
        tid_sql   = tarjeta_subquery(last4_row)
        lineas.append(
            f"INSERT INTO compras_msi "
            f"(usuario_id,tarjeta_id,descripcion,comercio,monto_total,total_pagos,pagos_hechos,cuota_mensual,fecha_inicio,activo) "
            f"SELECT {USUARIO_ID},{tid_sql},{v(row['descripcion'])},{v(row['comercio'])},"
            f"{row['monto_total']},{int(row['total_pagos'])},{int(row['pagos_hechos'])},"
            f"{row['cuota_mensual']},{v(row['fecha_inicio'])},TRUE "
            f"WHERE NOT EXISTS (SELECT 1 FROM compras_msi WHERE usuario_id={USUARIO_ID} "
            f"AND descripcion={v(row['descripcion'])} AND monto_total={row['monto_total']});"
        )
    sql_msi = '\n'.join(lineas)
    (OUT / 'insert_msi.sql').write_text(sql_msi)
    print(f'✅ {len(df_msi)} planes MSI → output/insert_msi.sql\n')
    print(sql_msi)